# GMM Classification with Test-Set Fit on SNGP Embeddings

Load saved SNGP embeddings, fit a label-assigned full-covariance GMM on the **CIFAR-10 test set**, and evaluate the fitted GMM on both the **test set** and the **training set**.

Reported metrics use clear split-specific naming:
- GMM posterior accuracy on test set
- GMM posterior accuracy on training set



In [1]:
from pathlib import Path
import os
import sys

import numpy as np

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "src").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError(f"Could not find repo root from {cwd}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


repo root: /w/20252/wjcai/uq/manygp


In [2]:
EMBEDDING_PATH = REPO_ROOT / "notebooks" / "gmm" / "sngp_embeddings_cifar10_sngp_epoch175_acc0.9323.npz"

if not EMBEDDING_PATH.exists():
    candidates = sorted((REPO_ROOT / "notebooks" / "gmm").glob("sngp_embeddings_*.npz"))
    if not candidates:
        raise FileNotFoundError("No sngp_embeddings_*.npz files found in notebooks/gmm")
    EMBEDDING_PATH = candidates[-1]

data = np.load(EMBEDDING_PATH, allow_pickle=True)

train_embeddings = data["train_embeddings"].astype(np.float64)
train_labels = data["train_labels"].astype(np.int64)
test_embeddings = data["test_embeddings"].astype(np.float64)
test_labels = data["test_labels"].astype(np.int64)
classes = data["classes"]
checkpoint_id = str(data["checkpoint_id"]) if "checkpoint_id" in data else EMBEDDING_PATH.stem.removeprefix("sngp_embeddings_")

print("embedding file:", EMBEDDING_PATH)
print("checkpoint id:", checkpoint_id)
print("train embeddings:", train_embeddings.shape)
print("test embeddings:", test_embeddings.shape)
print("classes:", classes.tolist())


embedding file: /w/20252/wjcai/uq/manygp/notebooks/gmm/sngp_embeddings_cifar10_sngp_epoch175_acc0.9323.npz
checkpoint id: cifar10_sngp_epoch175_acc0.9323
train embeddings: (50000, 128)
test embeddings: (10000, 128)
classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


Fit the GMM with labels fixed on the **test set**: `z = y_test`. The class prior `p(z)` is the empirical class frequency in the test set, and `p(x | z)` is a full-covariance Gaussian estimated from the test embeddings for that class.


In [3]:
def fit_label_assigned_gmm(x, y, num_classes, covariance_reg=1e-3):
    dim = x.shape[1]
    means = np.zeros((num_classes, dim), dtype=np.float64)
    covariances = np.zeros((num_classes, dim, dim), dtype=np.float64)
    priors = np.zeros(num_classes, dtype=np.float64)

    global_variance = np.var(x, axis=0).mean()
    reg = covariance_reg * max(global_variance, 1e-12)

    for cls in range(num_classes):
        class_x = x[y == cls]
        if len(class_x) == 0:
            raise ValueError(f"class {cls} has no samples")

        priors[cls] = len(class_x) / len(x)
        means[cls] = class_x.mean(axis=0)
        centered = class_x - means[cls]
        cov = centered.T @ centered / max(len(class_x) - 1, 1)
        covariances[cls] = cov + reg * np.eye(dim)

    return priors, means, covariances, reg


def logsumexp(a, axis=None, keepdims=False):
    a_max = np.max(a, axis=axis, keepdims=True)
    out = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out


def gaussian_logpdf_by_class(x, means, covariances):
    num_classes, dim = means.shape
    logpdf = np.empty((x.shape[0], num_classes), dtype=np.float64)
    log_2pi = dim * np.log(2.0 * np.pi)

    for cls in range(num_classes):
        chol = np.linalg.cholesky(covariances[cls])
        diff = (x - means[cls]).T
        solved = np.linalg.solve(chol, diff)
        mahalanobis = np.sum(solved * solved, axis=0)
        logdet = 2.0 * np.sum(np.log(np.diag(chol)))
        logpdf[:, cls] = -0.5 * (log_2pi + logdet + mahalanobis)

    return logpdf


def gmm_probabilities(x, priors, means, covariances):
    log_likelihood = gaussian_logpdf_by_class(x, means, covariances)
    log_joint = np.log(priors)[None, :] + log_likelihood
    log_px = logsumexp(log_joint, axis=1)
    posterior = np.exp(log_joint - log_px[:, None])

    return {
        "log_likelihood": log_likelihood,
        "log_joint": log_joint,
        "log_px": log_px,
        "posterior": posterior,
    }


num_classes = len(classes)
test_fit_priors, test_fit_means, test_fit_covariances, test_fit_covariance_reg = fit_label_assigned_gmm(
    test_embeddings,
    test_labels,
    num_classes=num_classes,
    covariance_reg=1e-3,
)

test_fit_test_gmm = gmm_probabilities(
    test_embeddings,
    test_fit_priors,
    test_fit_means,
    test_fit_covariances,
)
test_fit_train_gmm = gmm_probabilities(
    train_embeddings,
    test_fit_priors,
    test_fit_means,
    test_fit_covariances,
)

print("test-fit class priors:", np.round(test_fit_priors, 4))
print("test-fit covariance diagonal regularizer:", test_fit_covariance_reg)
print("test-fit log p(x) range on test set:", (test_fit_test_gmm["log_px"].min(), test_fit_test_gmm["log_px"].max()))
print("test-fit log p(x) range on training set:", (test_fit_train_gmm["log_px"].min(), test_fit_train_gmm["log_px"].max()))



test-fit class priors: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
test-fit covariance diagonal regularizer: 0.0005947019625565877
test-fit log p(x) range on test set: (np.float64(232.99158937280595), np.float64(301.3611840801706))
test-fit log p(x) range on training set: (np.float64(179.3021473960184), np.float64(301.33630751956724))


For each split, compute the class-posterior prediction `argmax p_gmm(y | x)` and report the corresponding accuracy.



In [4]:
test_set_posterior_pred = test_fit_test_gmm["posterior"].argmax(axis=1)
training_set_posterior_pred = test_fit_train_gmm["posterior"].argmax(axis=1)

gmm_posterior_accuracy_on_test_set = np.mean(test_set_posterior_pred == test_labels)
gmm_posterior_accuracy_on_training_set = np.mean(training_set_posterior_pred == train_labels)

print(f"GMM posterior accuracy on test set: {gmm_posterior_accuracy_on_test_set * 100:.2f}%")
print(f"GMM posterior accuracy on training set: {gmm_posterior_accuracy_on_training_set * 100:.2f}%")



GMM posterior accuracy on test set: 93.32%
GMM posterior accuracy on training set: 100.00%


In [5]:
# result_path = REPO_ROOT / "notebooks" / "gmm" / f"gmm_test_fit_results_{checkpoint_id}.npz"
# np.savez_compressed(
#     result_path,
#     test_fit_priors=test_fit_priors,
#     test_fit_means=test_fit_means,
#     test_fit_covariances=test_fit_covariances,
#     test_fit_covariance_reg=test_fit_covariance_reg,
#     test_fit_log_px_on_test_set=test_fit_test_gmm["log_px"],
#     test_fit_log_px_on_training_set=test_fit_train_gmm["log_px"],
#     test_fit_posterior_probs_on_test_set=test_fit_test_gmm["posterior"],
#     test_fit_posterior_probs_on_training_set=test_fit_train_gmm["posterior"],
#     test_set_posterior_pred=test_set_posterior_pred,
#     training_set_posterior_pred=training_set_posterior_pred,
#     gmm_posterior_accuracy_on_test_set=gmm_posterior_accuracy_on_test_set,
#     gmm_posterior_accuracy_on_training_set=gmm_posterior_accuracy_on_training_set,
#     train_labels=train_labels,
#     test_labels=test_labels,
#     classes=classes,
#     checkpoint_id=checkpoint_id,
#     embedding_path=str(EMBEDDING_PATH),
# )
# print("saved:", result_path)

